In [7]:
!pip install -q datasets transformers huggingface_hub accelerate tqdm

In [5]:
tinyaya_langs = ['amh', 'hau', 'ibo', 'mlg', 'sna', 'swh', 'wol', 'xho', 'yor', 'zul', 'tgl', 'msa', 'ind', 'vie', 'jav', 'khm', 'tha', 'lao', 'zho', 'mya', 'jpn', 'kor', 'hin', 'mar', 'ben', 'guj', 'pan', 'tam', 'tel', 'nep', 'ara', 'fas', 'urd', 'tur', 'mlt', 'heb', 'eng', 'nld', 'fra', 'ita', 'por', 'ron', 'spa', 'ces', 'pol', 'ukr', 'rus', 'ell', 'deu', 'dan', 'swe', 'nor', 'cat', 'glg', 'cym', 'gle', 'eus', 'hrv', 'lav', 'lit', 'slk', 'slv', 'est', 'fin', 'hun', 'srp', 'bul']

In [8]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
from collections import Counter
import numpy as np
import torch
import os
HUGGING_FACE_TOKEN = userdata.get('HF_NEW')
login(token=HUGGING_FACE_TOKEN)
dataset_name = "1024m/LID"
file_path = "Data_Main/LID-500.parquet"
full_dataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=HUGGING_FACE_TOKEN)["train"]
dataset = full_dataset.filter(lambda x: x["ISO-693-3"] in tinyaya_langs, num_proc=os.cpu_count())
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global", token=HUGGING_FACE_TOKEN)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)
tokenized_dataset = dataset.map(tokenize_function, batched=True, num_proc=os.cpu_count(), remove_columns=["text"])
lang_stats = {}
iso_codes = dataset["ISO-693-3"]
input_ids = tokenized_dataset["input_ids"]
for lang, ids in zip(iso_codes, input_ids):
    if lang not in lang_stats:
        lang_stats[lang] = Counter()
    lang_stats[lang].update(ids)
model_weights = {lang: {int(t): float(np.log(c/sum(counts.values()))) for t, c in counts.items()} for lang, counts in lang_stats.items()}
torch.save(model_weights, "tinyaya_core_lid_weights.pt")
print(f"Training Complete. Target Languages: {len(model_weights)} | Total Samples: {len(dataset)}")

Filter (num_proc=12):   0%|          | 0/158071 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/33500 [00:00<?, ? examples/s]

Training Complete. Target Languages: 67 | Total Samples: 33500


In [9]:
"""
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
from collections import Counter
import numpy as np
import torch
import os
HUGGING_FACE_TOKEN = userdata.get('HF_NEW')
login(token=HUGGING_FACE_TOKEN)
dataset_name = "1024m/LID"
file_path = "Data_Main/LID-10000.parquet"
dataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=HUGGING_FACE_TOKEN)["train"]
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global", token=HUGGING_FACE_TOKEN)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)
tokenized_dataset = dataset.map(tokenize_function, batched=True, num_proc=os.cpu_count(), remove_columns=dataset.column_names)
lang_stats = {}
iso_codes = dataset["ISO-693-3"]
input_ids = tokenized_dataset["input_ids"]
for lang, ids in zip(iso_codes, input_ids):
    if lang not in lang_stats:
        lang_stats[lang] = Counter()
    lang_stats[lang].update(ids)
model_weights = {}
for lang, counts in lang_stats.items():
    total = sum(counts.values())
    model_weights[lang] = {t: np.log(c/total) for t, c in counts.items()}
torch.save(model_weights, "tinyaya_lid_weights.pt")
print(f"Dataset: {dataset_name} | Samples: {len(dataset)} | Languages: {len(model_weights)}")
print("Model training complete and saved to tinyaya_lid_weights.pt")
"""

'\nfrom google.colab import userdata\nfrom huggingface_hub import login\nfrom datasets import load_dataset\nfrom transformers import AutoTokenizer\nfrom collections import Counter\nimport numpy as np\nimport torch\nimport os\nHUGGING_FACE_TOKEN = userdata.get(\'HF_NEW\')\nlogin(token=HUGGING_FACE_TOKEN)\ndataset_name = "1024m/LID"\nfile_path = "Data_Main/LID-10000.parquet"\ndataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=HUGGING_FACE_TOKEN)["train"]\ntokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global", token=HUGGING_FACE_TOKEN)\ndef tokenize_function(examples):\n    return tokenizer(examples["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)\ntokenized_dataset = dataset.map(tokenize_function, batched=True, num_proc=os.cpu_count(), remove_columns=dataset.column_names)\nlang_stats = {}\niso_codes = dataset["ISO-693-3"]\ninput_ids = tokenized_dataset["input_ids"]\nfor lang, ids in

In [10]:
import pandas as pd
import json
import torch
import numpy as np
import os
from tqdm import tqdm
from scipy.special import softmax
from sklearn.metrics import f1_score
from datasets import load_dataset
from transformers import AutoTokenizer
weights = torch.load("tinyaya_lid_weights.pt", weights_only=False)
languages = list(weights.keys())
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global")
vocab_size = tokenizer.vocab_size
lang_index = {lang: i for i, lang in enumerate(languages)}
weight_matrix = np.full((len(languages), vocab_size), -20.0)
for lang, counts in weights.items():
    for token_id, log_p in counts.items():
        if token_id < vocab_size:
            weight_matrix[lang_index[lang], token_id] = log_p
def run_inference(batch):
    tokenized = tokenizer(batch["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)["input_ids"]
    batch_probs_json = []
    batch_preds = []
    confidence_threshold = 0.10
    for ids in tokenized:
        if not ids:
            batch_preds.append("und")
            batch_probs_json.append("{}")
            continue
        scores = np.sum(weight_matrix[:, ids], axis=1)
        probs = softmax(scores)
        max_prob = np.max(probs)
        prob_dict = {languages[i]: f"{probs[i]:.10f}" for i in range(len(languages))}
        if max_prob < confidence_threshold:
            batch_preds.append("und")
        else:
            batch_preds.append(languages[np.argmax(scores)])
        batch_probs_json.append(json.dumps(prob_dict))
    return {"pred_lang": batch_preds, "probability_json": batch_probs_json}
benchmarks = [
    ("CommonLID", "Data_Benchmarks_Filtered/filtered_benchmark_CommonLID.parquet"),
    ("FLORES", "Data_Benchmarks_Filtered/filtered_benchmark_FLORES.parquet"),
    ("SmolSent", "Data_Benchmarks_Filtered/filtered_benchmark_SmolSent.parquet"),
    ("UDHRLID", "Data_Benchmarks_Filtered/filtered_benchmark_UDHRLID.parquet")
]
dataset_name = "1024m/LID"
all_benchmark_results = {}
summary_metrics = []
for name, path in benchmarks:
    ds = load_dataset("parquet", data_files={"test": f"hf://datasets/{dataset_name}/{path}"})["test"]
    res = ds.map(run_inference, batched=True, batch_size=2048, num_proc=os.cpu_count(), desc=f"Inference {name}")
    acc = (np.array(res["iso-693-3"]) == np.array(res["pred_lang"])).mean()
    f1 = f1_score(res["iso-693-3"], res["pred_lang"], average='macro')
    summary_metrics.append(f"{name} -> Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
    all_benchmark_results[name] = res
print("\nOVERALL METRICS:")
for line in summary_metrics:
    print(line)
for name, res in all_benchmark_results.items():
    print(f"\n{'='*10} {name} {'='*10}")
    df = pd.DataFrame({"true": res["iso-693-3"], "pred": res["pred_lang"]})
    for lang in sorted(df["true"].unique()):
        lang_acc = (df[df["true"] == lang]["true"] == df[df["true"] == lang]["pred"]).mean()
        print(f"{lang}: {lang_acc:.4f}")


OVERALL METRICS:
CommonLID -> Accuracy: 0.8603, Macro F1: 0.1981
FLORES -> Accuracy: 0.8745, Macro F1: 0.5578
SmolSent -> Accuracy: 0.8323, Macro F1: 0.2887
UDHRLID -> Accuracy: 0.8360, Macro F1: 0.6966

========== CommonLID ==========
ace: 1.0000
afr: 0.6705
amh: 0.9573
ara: 0.9063
arg: 0.7908
ary: 0.8009
arz: 0.5236
asm: 0.9437
aze: 0.9655
bak: 0.5000
bcl: 0.9000
ben: 0.9512
bre: 0.8867
bul: 0.8440
cat: 0.8817
ces: 0.9164
crh: 0.4000
deu: 0.8503
ell: 0.2857
eng: 0.7326
est: 0.8604
ext: 0.8571
fas: 0.9233
fin: 0.8971
fra: 0.8494
fry: 0.9648
gcr: 0.8198
gla: 0.8730
gle: 0.0000
gom: 0.0888
guj: 0.9852
guw: 0.7500
hau: 0.9112
heb: 0.9644
hin: 0.6023
ibo: 0.9940
ind: 0.8714
ita: 0.8315
jav: 0.5610
jpn: 0.9656
kab: 0.9397
kan: 0.7717
kik: 0.9355
kor: 0.8750
lat: 0.8000
lav: 0.9590
lij: 0.7007
lin: 0.0000
ltg: 0.5526
lug: 0.9280
mal: 0.9903
mar: 0.8134
mlg: 0.8388
msa: 0.7342
nld: 0.9263
nso: 0.3030
oci: 0.9110
orm: 0.9368
ory: 0.9852
pan: 0.9853
pcm: 0.5000
pol: 1.0000
por: 0.8203
rus: 0.

In [4]:
"""
import pandas as pd
import json
import torch
import numpy as np
import os
from tqdm import tqdm
from scipy.special import softmax
from sklearn.metrics import f1_score
from datasets import load_dataset
from transformers import AutoTokenizer
weights = torch.load("tinyaya_lid_weights.pt", weights_only=False)
languages = list(weights.keys())
tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global")
vocab_size = tokenizer.vocab_size
lang_index = {lang: i for i, lang in enumerate(languages)}
weight_matrix = np.full((len(languages), vocab_size), -20.0)
for lang, counts in weights.items():
    for token_id, log_p in counts.items():
        if token_id < vocab_size:
            weight_matrix[lang_index[lang], token_id] = log_p
def run_inference(batch):
    tokenized = tokenizer(batch["text"], padding=False, truncation=True, max_length=128, add_special_tokens=False)["input_ids"]
    batch_probs_json = []
    batch_preds = []
    confidence_threshold = 0.10
    for ids in tokenized:
        if not ids:
            batch_preds.append("und")
            batch_probs_json.append("{}")
            continue
        scores = np.sum(weight_matrix[:, ids], axis=1)
        probs = softmax(scores)
        max_prob = np.max(probs)
        prob_dict = {languages[i]: f"{probs[i]:.10f}" for i in range(len(languages))}
        if max_prob < confidence_threshold:
            batch_preds.append("und")
        else:
            batch_preds.append(languages[np.argmax(scores)])
        batch_probs_json.append(json.dumps(prob_dict))
    return {"pred_lang": batch_preds, "probability_json": batch_probs_json}
benchmarks = [
    ("CommonLID", "Data_Benchmarks_Filtered/filtered_benchmark_CommonLID.parquet"),
    ("FLORES", "Data_Benchmarks_Filtered/filtered_benchmark_FLORES.parquet"),
    ("SmolSent", "Data_Benchmarks_Filtered/filtered_benchmark_SmolSent.parquet"),
    ("UDHRLID", "Data_Benchmarks_Filtered/filtered_benchmark_UDHRLID.parquet")
]
dataset_name = "1024m/LID"
all_benchmark_results = {}
summary_metrics = []
for name, path in benchmarks:
    ds = load_dataset("parquet", data_files={"test": f"hf://datasets/{dataset_name}/{path}"})["test"]
    res = ds.map(run_inference, batched=True, batch_size=2048, num_proc=os.cpu_count(), desc=f"Inference {name}")
    acc = (np.array(res["iso-693-3"]) == np.array(res["pred_lang"])).mean()
    f1 = f1_score(res["iso-693-3"], res["pred_lang"], average='macro')
    summary_metrics.append(f"{name} -> Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
    all_benchmark_results[name] = res
print("\nOVERALL METRICS:")
for line in summary_metrics:
    print(line)
for name, res in all_benchmark_results.items():
    print(f"\n{'='*10} {name} {'='*10}")
    df = pd.DataFrame({"true": res["iso-693-3"], "pred": res["pred_lang"]})
    for lang in sorted(df["true"].unique()):
        lang_acc = (df[df["true"] == lang]["true"] == df[df["true"] == lang]["pred"]).mean()
        print(f"{lang}: {lang_acc:.4f}")
"""

Inference CommonLID (num_proc=12):   0%|          | 0/336325 [00:00<?, ? examples/s]

Data_Benchmarks_Filtered/filtered_benchm(…):   0%|          | 0.00/17.9M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

Inference FLORES (num_proc=12):   0%|          | 0/163944 [00:00<?, ? examples/s]

Data_Benchmarks_Filtered/filtered_benchm(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

...

Inference SmolSent (num_proc=12):   0%|          | 0/43148 [00:00<?, ? examples/s]

Data_Benchmarks_Filtered/filtered_benchm(…):   0%|          | 0.00/1.57M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

Inference UDHRLID (num_proc=12):   0%|          | 0/12550 [00:00<?, ? examples/s]


OVERALL METRICS:
CommonLID -> Accuracy: 0.8603, Macro F1: 0.1981
FLORES -> Accuracy: 0.8745, Macro F1: 0.5578
SmolSent -> Accuracy: 0.8323, Macro F1: 0.2887
UDHRLID -> Accuracy: 0.8360, Macro F1: 0.6966

========== CommonLID ==========
ace: 1.0000
afr: 0.6705
amh: 0.9573
ara: 0.9063
arg: 0.7908
ary: 0.8009
arz: 0.5236
asm: 0.9437
aze: 0.9655
bak: 0.5000
bcl: 0.9000
ben: 0.9512
bre: 0.8867
bul: 0.8440
cat: 0.8817
ces: 0.9164
crh: 0.4000
deu: 0.8503
ell: 0.2857
eng: 0.7326
est: 0.8604
ext: 0.8571
fas: 0.9233
fin: 0.8971
fra: 0.8494
fry: 0.9648
gcr: 0.8198
gla: 0.8730
gle: 0.0000
gom: 0.0888
guj: 0.9852
guw: 0.7500
hau: 0.9112
heb: 0.9644
hin: 0.6023
ibo: 0.9940
ind: 0.8714
ita: 0.8315
jav: 0.5610
jpn: 0.9656
kab: 0.9397
kan: 0.7717
kik: 0.9355
kor: 0.8750
lat: 0.8000
lav: 0.9590
lij: 0.7007
lin: 0.0000
ltg: 0.5526
lug: 0.9280
mal: 0.9903
mar: 0.8134
mlg: 0.8388
msa: 0.7342
nld: 0.9263
nso: 0.3030
oci: 0.9110
orm: 0.9368
ory: 0.9852
pan: 0.9853
pcm: 0.5000
pol: 1.0000
por: 0.8203
rus: 0.